In [1]:
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.speech_dataset import SpeechDataset

dataset = SpeechDataset(
    manifest_path=root / "metadata" / "manifest.csv",
    audio_root=(
        root
        / "data"
        / "raw"
        / "common_voice_beijing26"
        / "zh-CN-beijing"
        / "clips"
    ),
    split="train",
    sample_rate=16000
)

print("Dataset样本数:",len(dataset))

sample = dataset[0]

print("字段:",list(sample.keys()))
print("UID:",sample["uid"])
print("音频类型:",type(sample["audio"]))
print("音频形状:", tuple(sample["audio"].shape))
print("采样点数:", sample["length"])
print("文本:", sample["text"])

assert sample["audio"].ndim == 1
assert sample["audio"].dtype.name if False else True
assert sample["length"] == sample["audio"].numel()
assert sample["length"] > 0

print("SpeechDataset最小测试通过")

Dataset样本数: 2281
字段: ['audio', 'length', 'uid', 'text']
UID: common_voice_zh-CN_27897382
音频类型: <class 'torch.Tensor'>
音频形状: (101376,)
采样点数: 101376
文本: 大厦的主要租户包括美林集团及美国运通大型金融机构。
SpeechDataset最小测试通过


In [1]:
#使用真实音频测试speech dataset
import sys
from pathlib import Path
import torch

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.speech_dataset import SpeechDataset, collate_fn

dataset = SpeechDataset(
    manifest_path=root / "metadata" / "manifest.csv",
    audio_root=(
        root
        / "data"
        / "raw"
        / "common_voice_beijing26"
        / "zh-CN-beijing"
        / "clips"
    ),
    split="train",
    sample_rate=16000
)

print("Dataset样本数:", len(dataset))

#读取前20条样本
samples = [
    dataset[i]
    for i in range(min(20, len(dataset)))
]

#找到其中最短和最长的两条音频
short_sample = min(
    samples,
    key=lambda x: x["length"]
)

long_sample = max(
    samples,
    key=lambda x: x["length"],
)

print("短音频长度:", short_sample["length"])
print("长音频长度:", long_sample["length"])

#组成batch
batch = collate_fn([
    short_sample,
    long_sample
])

print("\nBatch结果")
print("audio shape:", batch["audio"].shape)
print("lengths:",batch["lengths"])
print("padding_mask shape:", batch["padding_mask"].shape)

print(
    "每条音频padding数量:",
    batch["padding_mask"].sum(dim=1)
)

print("UID:", batch["uid"])

Dataset样本数: 2281
短音频长度: 35136
长音频长度: 105408

Batch结果
audio shape: torch.Size([2, 105408])
lengths: tensor([ 35136, 105408])
padding_mask shape: torch.Size([2, 105408])
每条音频padding数量: tensor([70272,     0])
UID: ['common_voice_zh-CN_28374955', 'common_voice_zh-CN_28333336']


In [2]:
#验收检查
# 1. batch形状正确
assert batch["audio"].ndim == 2

#2.原始长度没有丢失
assert torch.equal(
    batch["lengths"],
    torch.tensor([
        short_sample["length"],
        long_sample["length"],
    ])
)

#3.mask与audio形状一致
assert (
    batch["padding_mask"].shape
    == batch["audio"].shape
)

#4.短音频有效区域不能被标记为padding
assert not batch["padding_mask"][0, :short_sample["length"]].any()

# 5. 短音频的填充区域必须全部为True
assert batch["padding_mask"][0, short_sample["length"]:].all()

# 6. 最长音频没有padding
assert not batch["padding_mask"][1].any()

# 7. padding位置的数值为0
assert torch.all(
    batch["audio"][batch["padding_mask"]] == 0
)

print("Collate全部验收通过")

Collate全部验收通过


In [3]:
#DataLoader冒烟测试
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

batch = next(iter(loader))

print("audio shape:", batch["audio"].shape)
print("lengths:", batch["lengths"])
print("padding_mask shape:", batch["padding_mask"].shape)
print("UID:", batch["uid"])

assert batch["audio"].shape[0] == 2
assert batch["lengths"].shape == (2,)
assert batch["padding_mask"].shape == batch["audio"].shape
assert len(batch["uid"]) == 2

print("DataLoader检查通过")

audio shape: torch.Size([2, 101376])
lengths: tensor([101376,  76608])
padding_mask shape: torch.Size([2, 101376])
UID: ['common_voice_zh-CN_27897382', 'common_voice_zh-CN_28304063']
DataLoader检查通过


In [3]:
#W6D7Aduio Utils验收
import sys
from pathlib import Path

import torch
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.audio_utils import (
    load_audio,
    to_mono,
    resample_audio,
    save_audio,
)

#1.读取原始WAV
wav_path = (
    root
    / "data"
    / "derived"
    / "w05_snr"
    / "demo_clean.wav"
)

info = sf.info(wav_path)

waveform, original_sr = load_audio(wav_path)

print("原始音频信息")
print("采样率:", original_sr)
print("声道数:", waveform.shape[0])
print("Waveform shape:", waveform.shape)
print("时长:", waveform.shape[-1] / original_sr)

#2.转为单声道
mono = to_mono(waveform)

print("\n单声道信息")
print("Shape:", mono.shape)

#3.重采样到16kHz
target_sr = 16000

resampled = resample_audio(
    mono,
    original_sr,
    target_sr
)

print("\n重采样后信息")
print("采样率:", target_sr)
print("Shape:", resampled.shape)
print("采样点数:", resampled.shape[-1])
print("时长:", resampled.shape[-1] / target_sr)

# 4. 检查时长是否保持
duration_before = mono.shape[-1] / original_sr
duration_after = resampled.shape[-1] / target_sr

assert abs(duration_before - duration_after) < 1 / target_sr

#5.保存重采样后的音频
output_path = (
    root
    /"results"
    /"w06d7_resampled_16k.wav"
)

save_audio(
    output_path,
    resampled,
    target_sr
)

# 6. 重新读取，验证写盘结果
reloaded, reloaded_sr = load_audio(output_path)

print("\n写盘后检查")
print("采样率:", reloaded_sr)
print("Shape:", reloaded.shape)

assert reloaded_sr == target_sr
assert reloaded.shape == resampled.shape

print("Audio Utils验收通过")

2026-10-31 04:53:57,527 - INFO - Loaded D:\Projects\speech-ai-learning\data\derived\w05_snr\demo_clean.wav,shape=torch.Size([1, 202752]), sr=32000
2026-10-31 04:53:57,550 - INFO - Saved D:\Projects\speech-ai-learning\results\w06d7_resampled_16k.wav
2026-10-31 04:53:57,568 - INFO - Loaded D:\Projects\speech-ai-learning\results\w06d7_resampled_16k.wav,shape=torch.Size([1, 101376]), sr=16000


原始音频信息
采样率: 32000
声道数: 1
Waveform shape: torch.Size([1, 202752])
时长: 6.336

单声道信息
Shape: torch.Size([1, 202752])

重采样后信息
采样率: 16000
Shape: torch.Size([1, 101376])
采样点数: 101376
时长: 6.336

写盘后检查
采样率: 16000
Shape: torch.Size([1, 101376])
Audio Utils验收通过


In [4]:
import matplotlib
matplotlib.use("Agg")  # 使用非交互式后端

import matplotlib.pyplot as plt
import numpy as np

audio_np = resampled.detach().cpu().squeeze(0).numpy()

time_axis = np.arange(len(audio_np)) / target_sr

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(time_axis, audio_np)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Waveform - 16kHz")

fig.tight_layout()

figure_path = root / "results" / "w06d7_waveform.png"
fig.savefig(figure_path, dpi=120)
plt.close(fig)

print("图片保存成功:", figure_path)

图片保存成功: D:\Projects\speech-ai-learning\results\w06d7_waveform.png


In [ ]:
#检查features函数
import inspect
import src.features as features

print("features.py:", features.__file__)

for name, func in inspect.getmembers(
    features,
    inspect.isfunction
):
    print(f"\n函数：{name}")
    print("参数：", inspect.signature(func))

features.py: D:\Projects\speech-ai-learning\src\features.py

函数：compute_log_magnitude_db
参数： (D: numpy.ndarray, top_db: float = 80.0) -> numpy.ndarray

函数：compute_stft
参数： (waveform: numpy.ndarray, sr: int, n_fft: int = 512, win_length: int = 400, hop_length: int = 160, center: bool = True) -> numpy.ndarray

函数：hz_to_mel
参数： (frequencies, *, scale='htk')

函数：waveform_to_mel
参数： (waveform, sr: int, *, n_fft: int = 1024, win_length: int = 800, hop_length: int = 160, window: str = 'hann', center: bool = True, pad_mode: str = 'constant', n_mels: int = 80, fmin: float = 0.0, fmax: float | None = None, htk: bool = False, norm: str = 'slaney', eps: float = 1e-10) -> numpy.ndarray


In [8]:
import numpy as np
import matplotlib.pyplot as plt

from src.audio_utils import load_audio
from src.features import (
    compute_stft,
    compute_log_magnitude_db,
    waveform_to_mel,
)

#1.读取之前重采样后的wav
wav_path = root / "results" / "w06d7_resampled_16k.wav"

waveform_tensor, sr = load_audio(wav_path)

assert sr == 16000

waveform = waveform_tensor.squeeze(0).numpy()

#2.统一特征参数
n_fft = 512
win_length = 400
hop_length = 160

#3.计算STFT
D = compute_stft(
    waveform,
    sr,
    n_fft=n_fft,
    win_length=win_length,
    hop_length=hop_length,
    center=True
)

#4.转换为dB
stft_db = compute_log_magnitude_db(
    D,
    top_db=80
)

#5.计算Log-Mel
log_mel = waveform_to_mel(
    waveform,
    sr,
    n_fft=n_fft,
    win_length=win_length,
    hop_length=hop_length,
    window="hann",
    center=True,
    pad_mode="constant",
    n_mels=80,
    fmin=0.0,
    fmax=sr /2,
    htk=False,
    norm="slaney"
)

# 6. 检查结果
print("Waveform shape:", waveform.shape)
print("STFT shape:", D.shape)
print("STFT dB shape:", stft_db.shape)
print("Log-Mel shape:", log_mel.shape)

print("STFT是否为复数:", np.iscomplexobj(D))
print("STFT是否全部有限:", np.isfinite(stft_db).all())
print("Log-Mel是否全部有限:", np.isfinite(log_mel).all())

assert D.shape == (257, 634)
assert stft_db.shape == D.shape
assert log_mel.shape == (80, 634)

print("特征提取检查通过")

2026-10-31 05:08:21,208 - INFO - Loaded D:\Projects\speech-ai-learning\results\w06d7_resampled_16k.wav,shape=torch.Size([1, 101376]), sr=16000


Waveform shape: (101376,)
STFT shape: (257, 634)
STFT dB shape: (257, 634)
Log-Mel shape: (80, 634)
STFT是否为复数: True
STFT是否全部有限: True
Log-Mel是否全部有限: True
特征提取检查通过


In [ ]:
#绘制三联图
import matplotlib.pyplot as plt

duration = len(waveform) / sr
time_axis = np.arange(len(waveform)) / sr

fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 9),
)

# Waveform
axes[0].plot(time_axis, waveform)
axes[0].set_title("Waveform")
axes[0].set_ylabel("Amplitude")

# STFT
im1 = axes[1].imshow(
    stft_db,
    origin="lower",
    aspect="auto",
    extent=[0, duration, 0, sr / 2],
)
axes[1].set_title("STFT Log-Magnitude (dB)")
axes[1].set_ylabel("Frequency (Hz)")
fig.colorbar(im1, ax=axes[1], label="dB")

# Log-Mel
im2 = axes[2].imshow(
    log_mel,
    origin="lower",
    aspect="auto",
    extent=[0, duration, 0, 80],
)
axes[2].set_title("Log-Mel Spectrogram")
axes[2].set_ylabel("Mel Bin")
axes[2].set_xlabel("Time (s)")
fig.colorbar(im2, ax=axes[2], label="Log-Mel")

fig.tight_layout()

output_path = root / "results" / "w06d7_features.png"

fig.savefig(output_path, dpi=150)
plt.close(fig)

print("图片已保存:", output_path)

图片已保存: D:\Projects\speech-ai-learning\results\w06d7_features.png


In [11]:
#数据隔离检查
import pandas as pd
from pathlib import Path

manifest_path = root / "metadata" / "manifest.csv"

manifest = pd.read_csv(manifest_path)

print("Manifest总样本数:", len(manifest))
print("\n字段:")
print(manifest.columns.tolist())

print("\nSplit统计:")
print(manifest["split"].value_counts())

print("\n各字段缺失数量:")
for col in [
    "parent_uid",
    "speaker_id",
    "session_uid",
    "audio_path",
]:
    if col in manifest.columns:
        print(f"{col}: {manifest[col].isna().sum()}")

if "source_id" in manifest.columns:
    print("\nSource ID数量:", manifest["source_id"].nunique())
    print("Source ID示例:")
    print(manifest["source_id"].dropna().unique()[:10])

Manifest总样本数: 2860

字段:
['audio_path', 'speaker_id', 'text', 'source_id', 'parent_uid', 'start_time', 'end_time', 'split']

Split统计:
split
train    2281
dev       409
test      170
Name: count, dtype: int64

各字段缺失数量:
parent_uid: 0
speaker_id: 0
audio_path: 0

Source ID数量: 1
Source ID示例:
<StringArray>
['common_voice_beijing26_zh-CN']
Length: 1, dtype: str


In [12]:
#检查跨split重叠
from itertools import combinations

# 统一split名称
manifest["_split"] = (
    manifest["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

splits = ["train", "dev", "test"]

# 检查三个split是否全部存在
assert set(splits).issubset(
    set(manifest["_split"].dropna().unique())
)

# 需要检查的字段
check_columns = [
    "parent_uid",
    "speaker_id",
    "session_uid",
    "audio_path",
]

results = {}

for col in check_columns:

    if col not in manifest.columns:
        print(f"\n{col}: 字段不存在，跳过")
        continue

    print(f"\n========== {col} ==========")

    values = {}

    for split in splits:

        series = (
            manifest.loc[
                manifest["_split"] == split, col
            ]
            .astype("string")
            .str.strip()
        )

        series = series.replace("", pd.NA).dropna()

        # Windows路径统一格式
        if col == "audio_path":
            series = (
                series.str.replace("\\", "/", regex=False)
                .str.lower()
            )

        values[split] = set(series)

        print(
            f"{split}: "
            f"{len(values[split])} 个有效唯一值"
        )

    results[col] = {}

    for a, b in combinations(splits, 2):

        overlap = values[a] & values[b]

        results[col][f"{a}-{b}"] = overlap

        print(
            f"{a} 与 {b} 重叠数量:",
            len(overlap)
        )

        if overlap:
            print("重叠示例:", list(overlap)[:5])

print("\n跨Split检查完成")


========== parent_uid ==========
train: 2281 个有效唯一值
dev: 409 个有效唯一值
test: 170 个有效唯一值
train 与 dev 重叠数量: 0
train 与 test 重叠数量: 0
dev 与 test 重叠数量: 0

========== speaker_id ==========
train: 27 个有效唯一值
dev: 33 个有效唯一值
test: 58 个有效唯一值
train 与 dev 重叠数量: 0
train 与 test 重叠数量: 0
dev 与 test 重叠数量: 0

session_uid: 字段不存在，跳过

========== audio_path ==========
train: 2281 个有效唯一值
dev: 409 个有效唯一值
test: 170 个有效唯一值
train 与 dev 重叠数量: 0
train 与 test 重叠数量: 0
dev 与 test 重叠数量: 0

跨Split检查完成
